# Data playground: Massive-STEPS New York

This notebook downloads the project's dataset from where it is published, checks that it is the exact file the project uses,
and takes a first look at it: size, fields, types, a few examples and basic data quality.

**Dataset.** Massive-STEPS New York: Foursquare check-ins grouped into trajectories, with venue names, categories and
coordinates. Each row is one check-in by one pseudonymous user.

**Where it lives.** The release is public on Hugging Face under the Apache 2.0 license. The notebook downloads it at a pinned
commit and verifies its SHA-256. No data is stored in this repository.

**Privacy.** Venue histories can identify a person, so this notebook never shows a user id, a trajectory id, an exact time or
two check-ins of the same person. The examples are single check-ins from five different users, at venues that 20 or more users
visited, with the year only.

## Provenance

| item | value |
|---|---|
| source | Hugging Face dataset `CRUISEResearchGroup/Massive-STEPS-New-York`, the release accompanying Wongso et al., *Massive-STEPS* (arXiv 2505.11239) |
| revision | commit `1e6af440a88ee54c83b1e30930290699e194a726` (dataset last modified 2025-09-26) |
| file | `new_york_checkins.csv`, the full check-in file; the release's own train / validation / test files split by trajectory for a different task and are not used |
| SHA-256 | `edc194f732cb236d841dbee775000c40d9e69bd9668c0aeebf266a159167b3a6`, identical to the project's working copy (first downloaded 2026-08-31) |
| license | Apache 2.0 (dataset card); upstream sources as stated by the authors: Semantic Trails (CC0 1.0) and Foursquare Open Source Places (Apache 2.0) |
| retrieved | the download cell below prints the time of this run |

In [1]:
# Setup: libraries and the pinned source. Versions are printed so the run can be reproduced (see requirements.txt).
import hashlib
import platform
import sys
from datetime import datetime, timezone

import huggingface_hub
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

REPO_ID = "CRUISEResearchGroup/Massive-STEPS-New-York"
REVISION = "1e6af440a88ee54c83b1e30930290699e194a726"   # pinned commit, so the file cannot change under us
FILENAME = "new_york_checkins.csv"
EXPECTED_SHA256 = "edc194f732cb236d841dbee775000c40d9e69bd9668c0aeebf266a159167b3a6"
MIN_EVENTS = 20          # the task keeps users with at least 20 check-ins (src/proxy/task.py)

print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| huggingface_hub", huggingface_hub.__version__)

python 3.12.7 | pandas 2.3.1 | numpy 2.5.2 | huggingface_hub 0.36.0


In [2]:
# Retrieve: download the file from the Hub at the pinned commit (cached outside the repository), verify its checksum, load it.
path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, repo_type="dataset", revision=REVISION)
sha = hashlib.sha256(open(path, "rb").read()).hexdigest()
assert sha == EXPECTED_SHA256, f"checksum mismatch: {sha}"
print("retrieved", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"), "| SHA-256 verified")

df = pd.read_csv(path, parse_dates=["timestamp"])
print(f"{len(df):,} rows x {df.shape[1]} columns")

retrieved 2026-10-06 00:35 UTC | SHA-256 verified


272,368 rows x 15 columns


## Size, fields and types

In [3]:
# Basic information: the schema, the number of users, venues and trajectories, and the time span.
schema = pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "distinct": df.nunique()})
display(schema)
print(f"users {df.user_id.nunique():,} | venues {df.venue_id.nunique():,} | trajectories {df.trail_id.nunique():,}")
print("time span:", df.timestamp.min().date(), "to", df.timestamp.max().date())
print(f"memory in pandas: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB")

,dtype,non_null,distinct
trail_id,object,272368,92041
user_id,int64,272368,6929
venue_id,int64,272368,45804
latitude,float64,252534,39533
longitude,float64,252534,38675
name,object,252534,33464
address,object,238135,28358
venue_category,object,272368,585
venue_category_id,object,272368,585
venue_category_id_code,int64,272368,585


users 6,929 | venues 45,804 | trajectories 92,041
time span: 2012-04-03 to 2018-10-19


memory in pandas: 135 MB


## A few examples

One check-in from each of five different users, drawn with a fixed seed from venues that 20 or more users visited. Ids,
coordinates and exact times are dropped; the year is kept because the release spans two periods (see below).

In [4]:
# Examples that follow the privacy rule above: different users, popular venues, no ids, year only.
venue_users = df.groupby("venue_id").user_id.nunique()
popular = df[df.venue_id.isin(venue_users[venue_users >= 20].index)]
one_per_user = popular.groupby("user_id").sample(n=1, random_state=0)
examples = one_per_user.sample(n=5, random_state=0)
display(examples.assign(year=examples.timestamp.dt.year)[["name", "venue_category", "venue_city", "year"]].reset_index(drop=True))

,name,venue_category,venue_city,year
0,Books Kinokuniya,Bookstore,Hell's Kitchen,2012
1,"Long Island City, NY",Neighborhood,Long Island City,2012
2,The Stumble Inn,Bar,Manhattan,2012
3,Otto Enoteca Pizzeria,Pizza Place,East Village,2012
4,Metropolitan Pavilion,Event Space,East Village,2012


## First look: data quality and coverage

In [5]:
# Missing values per column, as counts and shares.
missing = pd.DataFrame({"missing": df.isna().sum(), "share": df.isna().mean().round(4)})
display(missing[missing.missing > 0])

# Private places (homes and similar) carry no name or coordinates in the release.
print(f"check-ins without a venue name: {df.name.isna().mean():.1%}")

,missing,share
latitude,19834,0.0728
longitude,19834,0.0728
name,19834,0.0728
address,34233,0.1257


check-ins without a venue name: 7.3%


In [6]:
# Duplicates: identical rows, and the same user at the same venue at the same minute.
print("identical rows:", int(df.duplicated().sum()))
print("same user, venue and timestamp:", int(df.duplicated(["user_id", "venue_id", "timestamp"]).sum()))

# Ranges: coordinates should fall around New York City; timestamps inside the two collection periods.
display(df[["latitude", "longitude"]].describe().loc[["min", "max"]])
# A loose New York box (latitude 40.4 to 41.0, longitude -74.3 to -73.6): how many located check-ins fall outside it?
located = df.dropna(subset=["latitude", "longitude"])
outside = ~(located.latitude.between(40.4, 41.0) & located.longitude.between(-74.3, -73.6))
print(f"located check-ins outside the New York box: {int(outside.sum()):,} ({outside.mean():.2%}), "
      f"at {located[outside].venue_id.nunique():,} venues")

identical rows: 0
same user, venue and timestamp: 0


,latitude,longitude
min,35.590040,-122.407023
max,45.025688,-71.057281


located check-ins outside the New York box: 38 (0.02%), at 13 venues


In [7]:
# Two collection periods: trajectory ids start with 2013 (check-ins of 2012-13) or 2018 (check-ins of 2017-18).
df["period"] = df.trail_id.str[:4].map({"2013": "2012-13", "2018": "2017-18"})
by_period = df.groupby("period").agg(check_ins=("user_id", "size"), users=("user_id", "nunique"),
                                     venues=("venue_id", "nunique"), first=("timestamp", "min"), last=("timestamp", "max"))
by_period["share_of_check_ins"] = (by_period.check_ins / len(df)).round(3)
display(by_period)
periods_per_user = df.groupby("user_id").period.nunique()
print("users with check-ins in both periods:", int((periods_per_user == 2).sum()))

,check_ins,users,venues,first,last,share_of_check_ins
period,,,,,,
2012-13,233398,5426,39231,2012-04-03 18:00:00,2013-09-16 23:20:00,0.857
2017-18,38970,1632,12936,2017-10-03 14:58:00,2018-10-19 20:53:00,0.143


users with check-ins in both periods: 129


In [8]:
# Activity per user: most users have few check-ins, and the task keeps only users with at least MIN_EVENTS.
per_user = df.groupby("user_id").size()
display(per_user.describe(percentiles=[0.1, 0.5, 0.9]).round(1).to_frame("check-ins per user"))
eligible = per_user >= MIN_EVENTS
print(f"users with at least {MIN_EVENTS} check-ins: {eligible.sum():,} of {len(per_user):,} ({eligible.mean():.1%}), "
      f"holding {per_user[eligible].sum() / per_user.sum():.1%} of all check-ins")

,check-ins per user
count,6929.0
mean,39.3
std,79.5
min,6.0
10%,8.0
50%,19.0
90%,81.0
max,3193.0


users with at least 20 check-ins: 3,371 of 6,929 (48.7%), holding 85.1% of all check-ins


In [9]:
# Coverage: the ten most common venue categories, as shares of all check-ins.
display(df.venue_category.value_counts(normalize=True).head(10).round(3).to_frame("share"))

,share
venue_category,
Metro Station,0.041
Bar,0.037
Coffee Shop,0.035
Home (private),0.031
Office,0.031
Park,0.025
Neighborhood,0.019
Hotel,0.018
Train Station,0.018


## What I noticed

Numbers below are the outputs printed above.

- **Two collection periods of very different size.** 2012-13 holds 85.7% of the check-ins (233,398 from 5,426 users) and
  2017-18 only 14.3% (38,970 from 1,632 users); 129 users appear in both. This matters for the task: a candidate pool drawn
  from the whole release mixes venues from both periods, so on a 2017-18 decision a wrong option can be recognised by its
  period alone, so the project draws the wrong options for 2017-18 decisions from the same period.
- **Most users are light.** The median user has 19 check-ins. 3,371 of 6,929 users (48.7%) reach the task's floor of 20, and
  they hold 85.1% of all check-ins, so results describe active users, not every user.
- **Private places carry no name or location.** 7.3% of check-ins have no venue name or coordinates and 12.6% have no address;
  "Home (private)" alone is 3.1% of all check-ins.
- **A few locations fall outside New York.** 38 located check-ins (0.02%) at 13 venues lie outside a loose New York box, with
  latitudes from 35.6 to 45.0 and longitudes from -122.4 to -71.1, so the release's city label is not a guarantee of location.
  No fix yet; the share is too small to move the task.
- **No duplicates.** No identical rows and no repeated user, venue and minute.

## Reproduce

```
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute --inplace playground.ipynb
```

Run with Python 3.12.7. The download needs no credentials; the file is cached by `huggingface_hub` outside the repository.